# 03 — Engagement & Monetization KPIs

Daily and monthly engagement, cohort retention and cumulative monetization. MAU is distinct players in a calendar month; stickiness uses average daily DAU divided by same-month MAU.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
ROOT = Path.cwd()
if not (ROOT / 'data').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.analytics import monetization_by_day
DATA = ROOT / 'data'
players = pd.read_csv(DATA / 'players.csv', parse_dates=['install_date'])
sessions = pd.read_csv(DATA / 'sessions.csv', parse_dates=['session_start'])
transactions = pd.read_csv(DATA / 'iap_transactions.csv', parse_dates=['transaction_timestamp'])
players = players.loc[players.acquisition_channel.ne('internal_qa')]
sessions = sessions.loc[sessions.player_id.isin(players.player_id)].copy()
sessions['activity_date'] = sessions.session_start.dt.date
sessions['month'] = sessions.session_start.dt.to_period('M')
dau = sessions.groupby('activity_date').player_id.nunique().rename('DAU')
mau = sessions.groupby('month').player_id.nunique().rename('MAU')
avg_dau = sessions.groupby('month').apply(lambda g:g.groupby(g.session_start.dt.date).player_id.nunique().mean(), include_groups=False).rename('average_DAU')
engagement = pd.concat([avg_dau,mau,(avg_dau/mau).rename('stickiness')],axis=1)
display(engagement)

## Cumulative ARPU / LTV and payer metrics

Cumulative LTV here is observed 30-day IAP revenue per assigned player (including zero spenders), not extrapolated lifetime value.

In [ ]:
ltv = monetization_by_day(players, transactions)
display(ltv.pivot(index='day', columns='variant', values='cumulative_ltv'))
display(ltv.loc[ltv.day.eq(30), ['variant','revenue','arpu','cumulative_ltv','arppu','payer_rate']])

## Weekly cohort retention

In [ ]:
cohorts = players.assign(install_week=players.install_date.dt.to_period('W'))
events = sessions.merge(cohorts[['player_id','install_date','install_week']],on='player_id')
events['week_age'] = ((events.session_start.dt.normalize()-events.install_date.dt.normalize()).dt.days // 7)
active = events.drop_duplicates(['player_id','week_age']).groupby(['install_week','week_age']).player_id.nunique().unstack()
cohort_size = cohorts.groupby('install_week').player_id.nunique()
display((active.div(cohort_size,axis=0)*100).round(1))